In [3]:
import pandas as pd

In [22]:
# Exercice 1 : Lecture.

# a) Lis commandes.csv sans préciser sep. Que se passe-t-il ? Explique pourquoi, en t'aidant de la ligne de la commande O06.
# commandes = pd.read_csv('commandes.csv') # Cela donne une erreur 
# le chargement plante sur la ligne 7 qui correspond à O06 car on attend un champ et la virgule fait qu'il y a deux champs.
# b) 🧮 Lis-le correctement, tout en texte. Donne shape et la liste des colonnes.
commandes = pd.read_csv('commandes.csv', sep=';', dtype=str)
print('Le shape : ',commandes.shape)
print('La liste des colonnes : ', list(commandes.columns))

Le shape :  (12, 6)
La liste des colonnes :  ['order_id', 'client_id', 'date_commande', 'montant', 'statut', 'canal']


In [7]:
# Exercice 2 : Premier diagnostic 🧮

#a) Donne le nombre de NaN par colonne.
print(commandes.isna().sum())
#b) Donne la répartition des valeurs de statut. Quel problème vois-tu ?
commandes['statut'].value_counts()
# Je remarque qu'il y a une valeur COMPLETED tout en masjuscule et 9 valeurs completed en minuscule il faut normaliser le majuscule

order_id         0
client_id        0
date_commande    0
montant          1
statut           0
canal            1
dtype: int64


statut
completed    9
pending      1
cancelled    1
COMPLETED    1
Name: count, dtype: int64

In [8]:
# Exercice 3 : Doublons 🧮

# Combien y a-t-il de lignes entièrement identiques ? 
ligne_identique = commandes.duplicated().sum()
print('ligne_identique :',ligne_identique)
# Combien de doublons sur order_id ? 
ligne_identique_order_id = commandes.duplicated(subset=['order_id']).sum()
print('ligne_identique_order_id :', ligne_identique_order_id)
# Supprime-les en affichant le nombre de lignes supprimées et le nombre de lignes restantes.
avant = len(commandes)
commandes_propres = commandes.drop_duplicates( subset=['order_id'], keep='first')
print("doublons supprimés :", avant - len(commandes_propres)) 
print("Nombre de ligne restant :", len(commandes_propres)) 

ligne_identique : 1
ligne_identique_order_id : 1
doublons supprimés : 1
Nombre de ligne restant : 11


In [9]:
#Exercice 4 : Nettoyage et conversion 🧮
df = commandes_propres.copy()
#a) Nettoie statut (espaces et minuscules), puis vérifie avec value_counts.
df['statut'] = df['statut'].str.strip().str.lower()
#b) Convertis montant en nombre dans une nouvelle colonne montant_num, en gérant d'abord la virgule.
df['montant'] = df['montant'].str.replace(',','.',regex=False)
df['montant_num'] = pd.to_numeric(df['montant'], errors='coerce')
# Combien de valeurs non vides sont devenues NaN, et lesquelles (order_id) ?
print(df.loc[(df["montant"].notna()) & (df["montant_num"].isna()), ["order_id", "montant", "montant_num"]])
#c) Convertis date_commande en date, en gérant les deux formats présents dans le fichier. 
d1 = pd.to_datetime(df["date_commande"], format="%Y-%m-%d", errors="coerce")
d2 = pd.to_datetime(df["date_commande"], format="%d/%m/%Y", errors="coerce")
df["date_ok"] = d1.fillna(d2)          # prend d1, sinon d2
# Combien de dates sont invalides, et lesquelles ? Pourquoi sont-elles invalides ?
print('\nNombre de date invalide : ', df["date_ok"].isna().sum())
print('\n',df.loc[df["date_ok"].isna(), ["date_commande", "date_ok"]])
#le 31 septembre n'est pas une date réelle c'est la cause

   order_id montant  montant_num
10      O10     abc          NaN

Nombre de date invalide :  1

   date_commande date_ok
9    2026-09-31     NaT


In [30]:
# Exercice 5 : Filtre 🧮
df_exo5 = df.copy()
# Après le nettoyage, liste les order_id des commandes completed d'octobre 2026.
df_exo5.loc[(df_exo5['statut']=='completed') & (df_exo5['date_commande']<='2026-10-31') & (df_exo5['date_commande']>='2026-10-01'), ['order_id']]


,order_id
0,O01
1,O02
5,O06
6,O07
10,O10
11,O11


In [31]:
#Exercice 6 : Regroupement 🧮
df_exo6 = df.copy()
# a) Sur les commandes de l'exercice 5, calcule par canal le nombre de commandes et le chiffre d'affaires, en gardant le canal vide.
res = (df_exo6.groupby("canal", dropna=False)
         .agg(nb_commandes=("order_id", "size"),
              ca=("montant_num", "sum"))
         .reset_index()
         .sort_values("ca", ascending=False))
print('AVEC DROPNA_FALSE :\n',res)
# b) Refais le calcul sans dropna=False. Quelle commande disparaît, et de quel montant ?
res_sans_dropna_false = (df_exo6.groupby("canal", dropna=True)
         .agg(nb_commandes=("order_id", "size"),
              ca=("montant_num", "sum"))
         .reset_index()
         .sort_values("ca", ascending=False))
print('\nSANS DROPNA FALSE : \n', res_sans_dropna_false)
# c) Une valeur du résultat devrait t'alerter d'un point de vue qualité. Laquelle, et quelle action proposerais-tu (chapitre 1) ?
# c'est la commande order_id, son montant est négatif -20.0 il faut la consercer et mettre NULL et alerter jusqu'à ce qu'on ait une valeur fiable

AVEC DROPNA_FALSE :
   canal  nb_commandes     ca
0   app             4  684.0
1   web             6  375.0
2   NaN             1   24.9

SANS DROPNA FALSE : 
   canal  nb_commandes     ca
0   app             4  684.0
1   web             6  375.0


In [32]:
#Exercice 7 : Merge 🧮
df_exo7 = commandes_propres.copy() # issu de l'exo 3

# a) Lis clients.json. Fais un merge LEFT des commandes (dédoublonnées à l'exercice 3) avec les clients sans dédoublonner les clients. 
# Combien de lignes obtiens-tu ? Pourquoi ?
clients = pd.read_json('clients.json')
m1 = commandes_propres.merge(clients, on='client_id', how='left', indicator=True)
print(m1)
# b) Dédoublonne les clients sur client_id, refais le merge avec indicator=True, et trouve les commandes orphelines.
clients = clients.drop_duplicates( subset=['client_id'], keep='first')
m2 = commandes_propres.merge(clients, on='client_id', how='left', indicator=True)
print(m2)

   order_id client_id date_commande montant     statut canal      nom  \
0       O01        K1    2026-10-01   80.00  completed   web   Diallo   
1       O02        K2    2026-10-02   35.00  completed   app   Martin   
2       O02        K2    2026-10-02   35.00  completed   app   Martin   
3       O03        K1    05/10/2026  210.00  completed   web   Diallo   
4       O04        K5    2026-10-05     NaN    pending   web    Petit   
5       O05        K3    2026-10-12  150.00  cancelled   app   Nguyen   
6       O06        K2    2026-10-20   24,90  completed   NaN   Martin   
7       O06        K2    2026-10-20   24,90  completed   NaN   Martin   
8       O07        K6    2026-10-31  499.00  COMPLETED   app   Moreau   
9       O08        K3    2026-11-01   45.00  completed   web   Nguyen   
10      O09        K9    2026-09-31   60.00  completed   web      NaN   
11      O10        K4    2026-10-15     abc  completed   app  Bernard   
12      O11        K2    2026-10-18  -20.00  comple

In [28]:
# Exercice 8 : Export
# Enregistre tes commandes nettoyées dans staging_commandes.csv, sans l'index. Ouvre le fichier pour vérifier.
df.to_csv("staging_commandes.csv", index=False)